<a href="https://colab.research.google.com/github/dchakazira/Daniel-s_Portfolio/blob/main/colab_notebooks/AIGoat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIGoat on Google Colab

Runs [AIGoat](https://github.com/AISecurityConsortium/AIGoat) end-to-end inside a Colab runtime. Installs Ollama, clones the repo, installs Python + npm deps, applies the database migrations, seeds the SQLite DB, starts the FastAPI backend and React frontend in the background, and exposes the app through Colab's built-in port proxy.

## Before you run

- **Runtime**: the default CPU runtime works. A GPU runtime (T4) makes Mistral dramatically faster — switch via `Runtime → Change runtime type` before running.
- **RAM**: you need at least ~12 GB free. Mistral alone is ~4.5 GB; embeddings + backend + Node dev server take another few GB. The optional tool-calling model in step 7 (`qwen3.5:9b`, ~6.6 GB download) needs more, so use a GPU runtime for it.
- **First run is slow.** Expect ~3–5 min on the `pip install` cell and another several minutes on `ollama pull mistral` (it's a 4.5 GB download).
- **Which model?** Mistral is enough for the chat and RAG labs. The agent labs, the MCP host labs, and the Agentic Kill Chain work best with a model that supports native tool calls. Run the optional cell in step 7 to add one.
- **CPU inference is slow.** If you're on CPU-only, chat responses may take 30–60 seconds each.
- **One port for the app.** The frontend uses relative URLs and the React dev server (`frontend/src/setupProxy.js`) proxies `/api` and `/media` to the backend, so the app only needs the port 3000 address. The last cell also prints a port 8000 address for the Swagger docs.

Run the cells top-to-bottom. The last cell prints the app link on Colab's own domain. `localhost:3000` only exists inside the Colab VM, so it will not open from your browser.

## 1. Install Ollama

The Ollama daemon is started in step 6.

In [2]:
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information...

## 2. Clone AIGoat and set the working directory

`os.chdir` is used instead of `!cd` so subsequent Python cells inherit the directory.

In [16]:
import os, subprocess

REPO_DIR = '/content/AIGoat'
REPO_URL = 'https://github.com/AISecurityConsortium/AIGoat.git'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
os.makedirs('logs', exist_ok=True)

# Make the repo importable as a package (so 'from app.core.database import ...' works later).
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('cwd:', os.getcwd())

cwd: /content/AIGoat


## 3. Install Python dependencies

No virtualenv. Colab's own Python is used (AIGoat targets 3.11; its dependencies support 3.10 to 3.13). This step is slow on first run because `sentence-transformers`, `chromadb`, `nemoguardrails`, and `mcp` pull in a lot of transitive packages.

In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.7/593.7 kB 19.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 36.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 28.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 111.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 144.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 123.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 114.5 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Success

## 4. Check the frontend API settings

In the current repo `frontend/src/config/api.js` already uses relative URLs whenever the page is not opened on port 3000 or 3001, which is the case behind Colab's proxy. The React dev server then forwards `/api/…` and `/media/…` to `http://localhost:8000` through `frontend/src/setupProxy.js`. Net effect: one exposed port, everything works, and no patch is needed.

Older checkouts hardcoded `http://localhost:8000` as the fallback, so this cell still patches that case.

In [5]:
from pathlib import Path

p = Path('/content/AIGoat/frontend/src/config/api.js')
src = p.read_text()

# Older checkouts only
legacy = "process.env.REACT_APP_API_URL || 'http://localhost:8000'"

if 'resolveApiBase' in src:
    print('api.js resolves relative URLs by itself. No patch needed.')
elif "BASE_URL: ''," in src:
    print('api.js is already patched.')
elif legacy in src:
    p.write_text(src.replace(legacy, "''"))
    print('Patched', p)
else:
    raise RuntimeError('api.js API base not found. The repo layout may have changed.')

if not Path('/content/AIGoat/frontend/src/setupProxy.js').exists():
    print('Note: no setupProxy.js. Relying on the "proxy" field in package.json.')

api.js resolves relative URLs by itself. No patch needed.


## 5. Install frontend npm packages

Using `%%bash` so the `cd` is scoped to this cell.

In [6]:
%%bash
cd /content/AIGoat/frontend
npm install --silent --no-audit --no-fund 2>&1 | tail -20

## 6. Start the Ollama daemon (background)

`subprocess.Popen` keeps the daemon alive for the whole kernel session. A plain `!ollama serve &` would be killed when the cell finishes.

In [7]:
import subprocess, time
import urllib.request, urllib.error

ollama_log = open('/content/AIGoat/logs/ollama.log', 'wb')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2) as r:
            if r.status == 200:
                print('Ollama is up on :11434')
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama did not start — check logs/ollama.log')

Ollama is up on :11434


## 7. Pull the Mistral model

~4.5 GB download on first run. Output streams progress bars.

In [8]:
!ollama pull mistral
!ollama list


NAME              ID              SIZE      MODIFIED               
mistral:latest    6577803aa9a0    4.4 GB    Less than a second ago    


## 7b. (Optional) Add a tool-calling model

The agent labs, MCP host labs, and the Agentic Kill Chain need a model that makes native tool calls. Mistral does not, so those labs show no tool calls with it. This cell pulls `qwen3.5:9b` (~6.6 GB) and sets it as `ollama.agent_model` for the shop agent. In the Agentic Kill Chain workbench, also pick it in the **Model** dropdown.

Skip this cell if you only want the chat and RAG labs. Run it **before** step 10, because the backend reads the config at startup.

In [9]:
AGENT_MODEL = 'qwen3.5:9b'   # set to None to skip

if AGENT_MODEL:
    !ollama pull {AGENT_MODEL}
    from pathlib import Path
    cfg = Path('/content/AIGoat/config/config.yml')
    text = cfg.read_text()
    if '  agent_model: null' in text:
        cfg.write_text(text.replace('  agent_model: null', f'  agent_model: "{AGENT_MODEL}"'))
        print('config.yml: ollama.agent_model =', AGENT_MODEL)
    else:
        print('agent_model is already set. Check config/config.yml.')
else:
    print('Skipped. Chat and RAG labs only.')


config.yml: ollama.agent_model = qwen3.5:9b


## 8. Apply the database migrations

The schema is managed by Alembic (the same step `scripts/start.sh` runs). It also creates the lab tables for the Agentic Kill Chain.

In [10]:
!python -m alembic upgrade head
print('DB schema ready')

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 0001, Baseline schema matching the pre-Alembic ``create_all`` tables.
INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, Add knowledge_base_entries provenance / ACL columns.
INFO  [alembic.runtime.migration] Running upgrade 0002 -> 0003, Agent runs, steps, pending approvals, and lab_session surface token.
INFO  [alembic.runtime.migration] Running upgrade 0003 -> 0004, Per-user per-lab agent memory (D5).
INFO  [alembic.runtime.migration] Running upgrade 0004 -> 0005, Rename lab ids to OWASP LLM Top 10 2026 numbering (D18).
INFO  [alembic.runtime.migration] Running upgrade 0005 -> 0006, Support tickets for attacker/victim labs.
INFO  [alembic.runtime.migration] Running upgrade 0006 -> 0007, Support ticket conversation and attachments.
INFO  [alembic.runtime.migration] Running upgrade 0007 -> 0008

## 9. Seed demo data (idempotent)

Matches the counts the bash script checks for: 5 users (alice, bob, charlie, frank, admin), 20+ products, 9 challenges. It then runs the same two syncs as `scripts/start.sh` (support tickets and challenge metadata), which are safe to repeat.

In [11]:
import asyncio, subprocess
import nest_asyncio
nest_asyncio.apply()
from sqlalchemy import select, func
from app.core.database import async_session
from app.models import User, Product, Challenge

async def _needs_seed():
    async with async_session() as db:
        u = (await db.execute(select(func.count(User.id)))).scalar() or 0
        p = (await db.execute(select(func.count(Product.id)))).scalar() or 0
        c = (await db.execute(select(func.count(Challenge.id)))).scalar() or 0
        return u < 5 or p < 20 or c < 9

if asyncio.run(_needs_seed()):
    subprocess.run(['python', '-m', 'scripts.seed'], check=True, cwd='/content/AIGoat')
    print('Demo data seeded')
else:
    print('Database already has required data')

for flag in ('--sync-support', '--sync-challenges'):
    subprocess.run(['python', '-m', 'scripts.seed', flag], check=True, cwd='/content/AIGoat')
print('Support tickets and challenge metadata synced')

Demo data seeded
Support tickets and challenge metadata synced


## 10. Start the backend (FastAPI / uvicorn) in the background

In [12]:
import subprocess, time
import urllib.request, urllib.error

backend_log = open('/content/AIGoat/logs/backend.log', 'wb')
backend_proc = subprocess.Popen(
    [
        'python', '-m', 'uvicorn', 'app.main:app',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--log-level', 'info',
    ],
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    cwd='/content/AIGoat',
)

for _ in range(60):
    try:
        with urllib.request.urlopen('http://localhost:8000/docs', timeout=2) as r:
            if r.status < 500:
                print('Backend is up on :8000')
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    if backend_proc.poll() is not None:
        raise RuntimeError('Backend exited early — run the logs cell at the bottom to see why')
    time.sleep(1)
else:
    print('Backend slow to start — check logs/backend.log if the next cells fail')

Backend is up on :8000


## 11. Start the frontend (React dev server) in the background

`CI=true` suppresses CRA's interactive "port in use" prompt and disables the auto-browser-open.

In [13]:
import os, subprocess, time
import urllib.request, urllib.error

env = os.environ.copy()
env['PORT'] = '3000'
env['HOST'] = '0.0.0.0'
env['BROWSER'] = 'none'
env['CI'] = 'true'
env['DANGEROUSLY_DISABLE_HOST_CHECK'] = 'true'
env['WDS_SOCKET_PORT'] = '0'

frontend_log = open('/content/AIGoat/logs/frontend.log', 'wb')
frontend_proc = subprocess.Popen(
    ['npm', 'start'],
    stdout=frontend_log,
    stderr=subprocess.STDOUT,
    cwd='/content/AIGoat/frontend',
    env=env,
)

# CRA cold compile is slow — wait up to ~3 minutes.
for _ in range(180):
    try:
        with urllib.request.urlopen('http://localhost:3000/', timeout=2) as r:
            if r.status == 200:
                print('Frontend is up on :3000')
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    if frontend_proc.poll() is not None:
        raise RuntimeError('Frontend exited early — run the logs cell at the bottom to see why')
    time.sleep(1)
else:
    print('Frontend slow to compile — check logs/frontend.log if the preview URL 404s')

Frontend is up on :3000


## 12. Open AIGoat in a browser tab

Colab gives every kernel port its own HTTPS address on a Google-owned domain. `google.colab.kernel.proxyPort` returns that address, and this cell prints it as the link text, so what you see is where the link goes (not `localhost`).

- **Port 3000** is the app. The React dev server forwards `/api/…` and `/media/…` to the backend.
- **Port 8000** is printed only for the FastAPI Swagger page at `/docs`, which the dev server does not proxy.

The links only work in the browser that is signed in to this Colab session, and they stop working when the runtime disconnects. Re-run this cell to print them again.

In [14]:
from google.colab import output
from IPython.display import HTML, display

# Colab proxy URLs for the kernel ports (not localhost, which only exists inside the VM).
APP_URL = output.eval_js('google.colab.kernel.proxyPort(3000)').rstrip('/')
API_URL = output.eval_js('google.colab.kernel.proxyPort(8000)').rstrip('/')

links = [
    ('AIGoat application', APP_URL),
    ('API docs (Swagger)', f'{API_URL}/docs'),
    ('Agentic Kill Chain (sign in as admin)', f'{APP_URL}/challenges?killchain=1'),
]
display(HTML(''.join(
    f'<p><b>{label}:</b> <a href="{url}" target="_blank" rel="noopener">{url}</a></p>'
    for label, url in links
)))

print()
print('==========================================')
print('       AI Goat is running!                ')
print('==========================================')
print()
print('  App:      ', APP_URL)
print('  API docs: ', f'{API_URL}/docs')
print()
print('  Demo login:  alice / password123  (also bob, charlie, frank)')
print('  Admin login: admin / admin123  (staff: admin assistant, RAG, Kill Chain)')
print()
print('  These links belong to this Colab session. http://localhost:3000 will not work from your own machine.')



       AI Goat is running!                

  App:       https://3000-gpu-t4-s-usw1b0-2eonn8pnuat0d-b.us-west1.prod.r.colab.dev
  API docs:  https://8000-gpu-t4-s-usw1b0-2eonn8pnuat0d-b.us-west1.prod.r.colab.dev/docs

  Demo login:  alice / password123  (also bob, charlie, frank)
  Admin login: admin / admin123  (staff: admin assistant, RAG, Kill Chain)

  These links belong to this Colab session. http://localhost:3000 will not work from your own machine.


## 13. (Optional) Tail logs or shut everything down

Uncomment whatever you need. Process handles (`ollama_proc`, `backend_proc`, `frontend_proc`) are held in kernel memory from the earlier cells.

In [15]:
# !tail -n 80 /content/AIGoat/logs/backend.log
# !tail -n 80 /content/AIGoat/logs/frontend.log
# !tail -n 80 /content/AIGoat/logs/ollama.log

#frontend_proc.terminate()
# backend_proc.terminate()
# ollama_proc.terminate()